# 02 - Cold-Start Protocol

Build one deterministic, leakage-safe MovieLens-1M protocol for LightGCN,
EmerG, DGD, and the proposed ablations. Labels, item cohorts, validation tasks,
and Cold/Warm-Up A/B/C assignments are fixed here and reused by every model.

## Notebook Linkage and Boundaries

**Input:** Notebook 01's authenticated `ml1m-audit-v2` manifest and canonical
interactions, users, and items.

**Output:** A self-contained protocol bundle with tuning/final training rows,
validation and new-item task rows, indexed metadata, item cohorts, checks, and
a manifest consumed by notebooks 03 and 04.

Feature tokenization, model fitting, threshold selection, and test metrics stay
downstream. Raw-data acquisition and parsing stay in notebook 01.

## Trusted Notebook-01 Handoff

Locate the audit pointer locally or in a mounted Kaggle input. Require the v2
schema and PASS status, confine every path to its audit root, verify all hashes,
compare the immutable bundle manifest, and load exact declared dtypes.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import shutil
import uuid
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import pandas as pd
from IPython.display import Markdown, display


def show_records(records: Iterable[dict[str, Any]]) -> None:
    display(pd.DataFrame(list(records)))


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_inside(root: Path, relative_path: str) -> Path:
    resolved_root = root.resolve()
    resolved = (resolved_root / relative_path).resolve()
    resolved.relative_to(resolved_root)
    return resolved


def project_root(start: Path) -> Path:
    override = os.environ.get("COLDSTART_PROJECT_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate.resolve()
    return start.resolve()


EXECUTION_CONTEXT = "kaggle" if Path("/kaggle/input").exists() else "local"
PROJECT_ROOT = project_root(Path.cwd())
WORKSPACE_ROOT = Path(
    os.environ.get(
        "COLDSTART_WORKSPACE_ROOT",
        "/kaggle/working" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / ".notebook",
    )
).expanduser().resolve()
INPUT_ROOT = Path(
    os.environ.get(
        "COLDSTART_INPUT_ROOT",
        "/kaggle/input" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / "data",
    )
).expanduser().resolve()
ARTIFACT_ROOT = Path(
    os.environ.get("COLDSTART_ARTIFACT_ROOT", WORKSPACE_ROOT / "artifacts")
).expanduser().resolve()
AUDIT_RELATIVE_MANIFEST = Path("processed/ml-1m/audit-v2/manifest.json")
PROTOCOL_OUTPUT_ROOT = ARTIFACT_ROOT / "protocols" / "ml-1m" / "coldstart-v1"

show_records(
    [
        {
            "execution_context": EXECUTION_CONTEXT,
            "python": platform.python_version(),
            "input_root": str(INPUT_ROOT),
            "artifact_root": str(ARTIFACT_ROOT),
            "protocol_output_root": str(PROTOCOL_OUTPUT_ROOT),
        }
    ]
)

,execution_context,python,input_root,artifact_root,protocol_output_root
0,kaggle,3.12.13,/kaggle/input,/kaggle/working/artifacts,/kaggle/working/artifacts/protocols/ml-1m/cold...


In [2]:
def audit_candidates() -> list[tuple[Path, Path]]:
    candidates: list[tuple[Path, Path]] = []
    explicit = os.environ.get("COLDSTART_AUDIT_ROOT")
    roots = [Path(explicit).expanduser()] if explicit else []
    roots.extend([PROJECT_ROOT / ".notebook" / "artifacts", ARTIFACT_ROOT])

    for root in roots:
        pointer = root / AUDIT_RELATIVE_MANIFEST
        if pointer.is_file():
            candidates.append((root.resolve(), pointer.resolve()))
        direct = root / "manifest.json"
        if root.name == "audit-v2" and direct.is_file():
            candidates.append((root.parents[2].resolve(), direct.resolve()))

    if INPUT_ROOT.is_dir():
        for pointer in sorted(INPUT_ROOT.rglob("manifest.json")):
            if pointer.parent.name == "audit-v2":
                candidates.append((pointer.parents[3].resolve(), pointer.resolve()))

    unique: list[tuple[Path, Path]] = []
    seen: set[str] = set()
    for root, pointer in candidates:
        key = str(pointer)
        if key not in seen:
            seen.add(key)
            unique.append((root, pointer))
    return unique


def load_verified_audit(root: Path, pointer: Path) -> tuple[dict[str, Any], dict[str, Any]]:
    pointer_bytes = pointer.read_bytes()
    manifest = json.loads(pointer_bytes)
    required_header = {
        "dataset": "MovieLens-1M",
        "release": "ml-1m",
        "audit_schema_version": "ml1m-audit-v2",
        "audit_status": "PASS",
    }
    for key, expected in required_header.items():
        if manifest.get(key) != expected:
            raise ValueError(f"Invalid audit {key}: {manifest.get(key)!r}")

    bundle_manifest = resolve_inside(root, manifest["bundle_manifest"])
    if bundle_manifest.read_bytes() != pointer_bytes:
        raise ValueError("Audit pointer and immutable bundle manifest differ")

    for artifact in manifest["artifacts"].values():
        path = resolve_inside(root, artifact["path"])
        if not path.is_file() or sha256_file(path) != artifact["sha256"]:
            raise ValueError(f"Audit artifact verification failed: {artifact['path']}")

    tables: dict[str, Any] = {}
    for name in ("interactions", "users", "items"):
        artifact = manifest["artifacts"][name]
        schema = manifest["canonical_output_schemas"][name]
        table = pd.read_csv(
            resolve_inside(root, artifact["path"]),
            dtype=schema["read_csv_dtypes"],
        )
        if list(table.columns) != schema["columns"] or len(table) != artifact["rows"]:
            raise ValueError(f"Audit table contract failed: {name}")
        tables[name] = table
    return manifest, tables


AUDIT_ERRORS: list[str] = []
AUDIT_ROOT = None
AUDIT_POINTER = None
AUDIT_MANIFEST = None
AUDIT_TABLES = None
for candidate_root, candidate_pointer in audit_candidates():
    try:
        AUDIT_MANIFEST, AUDIT_TABLES = load_verified_audit(candidate_root, candidate_pointer)
        AUDIT_ROOT, AUDIT_POINTER = candidate_root, candidate_pointer
        break
    except Exception as error:
        AUDIT_ERRORS.append(f"{candidate_pointer}: {error}")

if AUDIT_MANIFEST is None:
    raise RuntimeError(
        "No valid notebook-01 audit bundle found. Set COLDSTART_AUDIT_ROOT. "
        + " | ".join(AUDIT_ERRORS)
    )

INTERACTIONS = AUDIT_TABLES["interactions"]
USERS = AUDIT_TABLES["users"]
ITEMS = AUDIT_TABLES["items"]
UPSTREAM_POINTER_SHA256 = sha256_file(AUDIT_POINTER)

show_records(
    [
        {
            "audit_bundle": AUDIT_MANIFEST["bundle_id"],
            "audit_pointer_sha256": UPSTREAM_POINTER_SHA256,
            "interactions": len(INTERACTIONS),
            "users": len(USERS),
            "items": len(ITEMS),
        }
    ]
)

,audit_bundle,audit_pointer_sha256,interactions,users,items
0,20260716T201142-7f59c2e22253,e751b7f304d9d1ed756ff41b3ef04542738e3e75bb188d...,1000209,6040,3883


## Protocol Configuration

Match the proposal and EmerG paper for labels, cohorts, and new-item phases:
ratings 4-5 are positive; old items have more than `N=200` interactions; new
items have strictly more than `3K=60` and fewer than 200; A/B/C are 20-row
chronological increments. Mirroring A/B/C on the paper's 20% held-out old-item
validation set is an explicit project extension for phase-specific selection.

In [3]:
@dataclass(frozen=True)
class ProtocolConfig:
    schema_version: str = "ml1m-coldstart-v1"
    positive_rating_min: int = 4
    old_item_threshold: int = 200
    warm_block_size: int = 20
    validation_fraction: float = 0.20
    protocol_seed: int = 2025
    validation_policy: str = "20% old items by seeded SHA-256 rank; mirror A/B/C/query"
    candidate_policy: str = "observed labeled query rows; no synthetic negatives"
    graph_edge_policy: str = "all phase-visible interactions, independent of label"


CONFIG = ProtocolConfig()
CONFIG_PAYLOAD = asdict(CONFIG)
CONFIG_HASH = hashlib.sha256(
    json.dumps(CONFIG_PAYLOAD, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()
show_records([{**CONFIG_PAYLOAD, "configuration_sha256": CONFIG_HASH}])

,schema_version,positive_rating_min,old_item_threshold,warm_block_size,validation_fraction,protocol_seed,validation_policy,candidate_policy,graph_edge_policy,configuration_sha256
0,ml1m-coldstart-v1,4,200,20,0.2,2025,20% old items by seeded SHA-256 rank; mirror A...,observed labeled query rows; no synthetic nega...,"all phase-visible interactions, independent of...",7851d74536c20d1b9799ed275aa7f2122559c9a785472d...


## Cohorts, Training Sets, and Warm-Up Tasks

`source_row` preserves notebook 01 order and breaks timestamp ties. Query rows
are stored once and reused unchanged in Cold, Warm A, Warm B, and Warm C.
Tuning excludes validation items; final refitting restores all old items.

In [4]:
INTERACTIONS = INTERACTIONS.reset_index().rename(columns={"index": "source_row"})
INTERACTIONS["source_row"] = INTERACTIONS["source_row"].astype("int64")
ITEM_COUNTS = INTERACTIONS.groupby("item_id", sort=True).size().reindex(
    ITEMS["item_id"], fill_value=0
).astype("int64")

OLD_ITEMS = set(ITEM_COUNTS[ITEM_COUNTS > CONFIG.old_item_threshold].index.astype(int))
NEW_ITEMS = set(
    ITEM_COUNTS[
        (ITEM_COUNTS > 3 * CONFIG.warm_block_size)
        & (ITEM_COUNTS < CONFIG.old_item_threshold)
    ].index.astype(int)
)


def validation_order(item_id: int) -> tuple[str, int]:
    score = hashlib.sha256(f"{CONFIG.protocol_seed}:{item_id}".encode()).hexdigest()
    return score, item_id


validation_count = round(len(OLD_ITEMS) * CONFIG.validation_fraction)
VALIDATION_ITEMS = set(sorted(OLD_ITEMS, key=validation_order)[:validation_count])
TRAIN_OLD_ITEMS = OLD_ITEMS - VALIDATION_ITEMS
PROTOCOL_ITEMS = OLD_ITEMS | NEW_ITEMS

ITEM_COHORTS = ITEMS[["item_id"]].copy()
ITEM_COHORTS["interaction_count"] = ITEM_COHORTS["item_id"].map(ITEM_COUNTS).astype("int64")
ITEM_COHORTS["cohort"] = "excluded_rated"
ITEM_COHORTS.loc[ITEM_COHORTS["interaction_count"].eq(0), "cohort"] = "unrated"
ITEM_COHORTS.loc[ITEM_COHORTS["item_id"].isin(TRAIN_OLD_ITEMS), "cohort"] = "old_train"
ITEM_COHORTS.loc[ITEM_COHORTS["item_id"].isin(VALIDATION_ITEMS), "cohort"] = "old_validation"
ITEM_COHORTS.loc[ITEM_COHORTS["item_id"].isin(NEW_ITEMS), "cohort"] = "new"
ITEM_COHORTS["cohort"] = ITEM_COHORTS["cohort"].astype("string")

protocol_item_ids = sorted(PROTOCOL_ITEMS)
protocol_user_ids = sorted(
    INTERACTIONS.loc[INTERACTIONS["item_id"].isin(PROTOCOL_ITEMS), "user_id"].unique()
)
item_to_idx = {item_id: index for index, item_id in enumerate(protocol_item_ids)}
user_to_idx = {user_id: index for index, user_id in enumerate(protocol_user_ids)}

ITEM_COHORTS["item_idx"] = ITEM_COHORTS["item_id"].map(item_to_idx).astype("Int64")
PROTOCOL_USERS = USERS[USERS["user_id"].isin(protocol_user_ids)].copy()
PROTOCOL_USERS["user_idx"] = PROTOCOL_USERS["user_id"].map(user_to_idx).astype("int64")
PROTOCOL_USERS = PROTOCOL_USERS.sort_values("user_idx").reset_index(drop=True)
PROTOCOL_ITEMS_METADATA = ITEMS[ITEMS["item_id"].isin(protocol_item_ids)].copy()
PROTOCOL_ITEMS_METADATA["item_idx"] = (
    PROTOCOL_ITEMS_METADATA["item_id"].map(item_to_idx).astype("int64")
)
PROTOCOL_ITEMS_METADATA = PROTOCOL_ITEMS_METADATA.sort_values("item_idx").reset_index(drop=True)


def add_common_fields(table: pd.DataFrame) -> pd.DataFrame:
    result = table.copy()
    result["user_idx"] = result["user_id"].map(user_to_idx).astype("int64")
    result["item_idx"] = result["item_id"].map(item_to_idx).astype("int64")
    result["label"] = result["rating"].ge(CONFIG.positive_rating_min).astype("int8")
    return result


TRAIN_COLUMNS = [
    "source_row", "user_id", "user_idx", "item_id", "item_idx", "rating", "timestamp", "label"
]
TUNING_TRAIN = add_common_fields(
    INTERACTIONS[INTERACTIONS["item_id"].isin(TRAIN_OLD_ITEMS)]
).sort_values("source_row")[TRAIN_COLUMNS].reset_index(drop=True)
FINAL_TRAIN = add_common_fields(
    INTERACTIONS[INTERACTIONS["item_id"].isin(OLD_ITEMS)]
).sort_values("source_row")[TRAIN_COLUMNS].reset_index(drop=True)


def build_tasks(item_ids: set[int]) -> pd.DataFrame:
    tasks = INTERACTIONS[INTERACTIONS["item_id"].isin(item_ids)].copy()
    tasks = tasks.sort_values(["item_id", "timestamp", "source_row"]).reset_index(drop=True)
    tasks["item_rank"] = tasks.groupby("item_id", sort=False).cumcount().astype("int64")
    tasks["role"] = pd.Series("query", index=tasks.index, dtype="string")
    k = CONFIG.warm_block_size
    tasks.loc[tasks["item_rank"] < 3 * k, "role"] = "warm_c"
    tasks.loc[tasks["item_rank"] < 2 * k, "role"] = "warm_b"
    tasks.loc[tasks["item_rank"] < k, "role"] = "warm_a"
    tasks = add_common_fields(tasks)
    return tasks[
        TRAIN_COLUMNS + ["item_rank", "role"]
    ]


VALIDATION_TASKS = build_tasks(VALIDATION_ITEMS)
EVALUATION_TASKS = build_tasks(NEW_ITEMS)

## Protocol Summary

Warm supports are cumulative at model time: Cold uses none, A uses `warm_a`, B
uses `warm_a + warm_b`, and C uses all three. Every phase scores the same query.

In [5]:
COHORT_SUMMARY = (
    ITEM_COHORTS.groupby("cohort", observed=True)
    .agg(items=("item_id", "size"), interactions=("interaction_count", "sum"))
    .reset_index()
)
TASK_SUMMARY = pd.concat(
    [
        VALIDATION_TASKS.groupby("role", observed=True).size().rename("rows").reset_index().assign(split="validation"),
        EVALUATION_TASKS.groupby("role", observed=True).size().rename("rows").reset_index().assign(split="evaluation"),
    ],
    ignore_index=True,
)
PROTOCOL_SUMMARY = {
    "tuning_train_rows": len(TUNING_TRAIN),
    "final_train_rows": len(FINAL_TRAIN),
    "validation_task_rows": len(VALIDATION_TASKS),
    "evaluation_task_rows": len(EVALUATION_TASKS),
    "evaluation_query_rows": int(EVALUATION_TASKS["role"].eq("query").sum()),
    "evaluation_query_positives": int(
        EVALUATION_TASKS.loc[EVALUATION_TASKS["role"].eq("query"), "label"].sum()
    ),
    "users": len(PROTOCOL_USERS),
    "protocol_items": len(PROTOCOL_ITEMS_METADATA),
}
display(COHORT_SUMMARY)
display(TASK_SUMMARY)
show_records([PROTOCOL_SUMMARY])

,cohort,items,interactions
0,excluded_rated,1331,30810
1,new,955,114869
2,old_train,1136,684728
3,old_validation,284,169802
4,unrated,177,0


,role,rows,split
0,query,152762,validation
1,warm_a,5680,validation
2,warm_b,5680,validation
3,warm_c,5680,validation
4,query,57569,evaluation
5,warm_a,19100,evaluation
6,warm_b,19100,evaluation
7,warm_c,19100,evaluation


,tuning_train_rows,final_train_rows,validation_task_rows,evaluation_task_rows,evaluation_query_rows,evaluation_query_positives,users,protocol_items
0,684728,854530,169802,114869,57569,23501,6040,2375


## Leakage and Coverage Tests

Fail publication unless item sets are disjoint, strict frequency rules and
expected MovieLens-1M counts hold, A/B/C budgets are exact, queries are fixed
and nonempty, ordering is total, and validation/new rows never enter tuning.

In [6]:
PROTOCOL_CHECKS: list[dict[str, Any]] = []


def check(name: str, condition: bool, observed: Any, expected: Any) -> None:
    PROTOCOL_CHECKS.append(
        {"check": name, "status": "PASS" if condition else "FAIL", "observed": observed, "expected": expected}
    )


def warm_blocks_valid(tasks: pd.DataFrame) -> bool:
    for role in ("warm_a", "warm_b", "warm_c"):
        counts = tasks[tasks["role"].eq(role)].groupby("item_id").size()
        if len(counts) != tasks["item_id"].nunique() or not counts.eq(CONFIG.warm_block_size).all():
            return False
    return True


def task_partition_valid(tasks: pd.DataFrame, item_ids: set[int]) -> bool:
    expected = INTERACTIONS[INTERACTIONS["item_id"].isin(item_ids)].sort_values(
        ["item_id", "timestamp", "source_row"]
    )
    k = CONFIG.warm_block_size
    expected_roles = tasks["item_rank"].map(
        lambda rank: "warm_a" if rank < k else "warm_b" if rank < 2 * k else "warm_c" if rank < 3 * k else "query"
    )
    contiguous_ranks = tasks.groupby("item_id")["item_rank"].apply(
        lambda ranks: ranks.tolist() == list(range(len(ranks)))
    ).all()
    return (
        tasks["source_row"].is_unique
        and tasks["source_row"].tolist() == expected["source_row"].tolist()
        and contiguous_ranks
        and tasks["role"].astype(str).tolist() == expected_roles.tolist()
        and tasks[tasks["role"].eq("query")].groupby("item_id").size().gt(0).all()
    )


def labels_valid(table: pd.DataFrame) -> bool:
    return bool(
        (table["label"] == table["rating"].ge(CONFIG.positive_rating_min).astype("int8")).all()
    )


def indices_valid() -> bool:
    user_map_valid = (
        PROTOCOL_USERS["user_id"].is_unique
        and PROTOCOL_USERS["user_idx"].tolist() == list(range(len(PROTOCOL_USERS)))
    )
    item_map_valid = (
        PROTOCOL_ITEMS_METADATA["item_id"].is_unique
        and PROTOCOL_ITEMS_METADATA["item_idx"].tolist()
        == list(range(len(PROTOCOL_ITEMS_METADATA)))
    )
    table_maps_valid = all(
        table["user_id"].map(user_to_idx).equals(table["user_idx"])
        and table["item_id"].map(item_to_idx).equals(table["item_idx"])
        for table in (TUNING_TRAIN, FINAL_TRAIN, VALIDATION_TASKS, EVALUATION_TASKS)
    )
    return bool(user_map_valid and item_map_valid and table_maps_valid)


cohort_counts = ITEM_COHORTS.set_index("cohort").groupby(level=0)["item_id"].count().to_dict()
check("old item count", len(OLD_ITEMS) == 1420, len(OLD_ITEMS), 1420)
check("new item count", len(NEW_ITEMS) == 955, len(NEW_ITEMS), 955)
check("old validation item count", len(VALIDATION_ITEMS) == 284, len(VALIDATION_ITEMS), 284)
check("old tuning item count", len(TRAIN_OLD_ITEMS) == 1136, len(TRAIN_OLD_ITEMS), 1136)
check("excluded rated item count", cohort_counts.get("excluded_rated") == 1331, cohort_counts.get("excluded_rated"), 1331)
check("unrated item count", cohort_counts.get("unrated") == 177, cohort_counts.get("unrated"), 177)
check("item cohorts disjoint", not (OLD_ITEMS & NEW_ITEMS) and not (TRAIN_OLD_ITEMS & VALIDATION_ITEMS), True, True)
check("tuning rows", len(TUNING_TRAIN) == 684728, len(TUNING_TRAIN), 684728)
check("final old-item rows", len(FINAL_TRAIN) == 854530, len(FINAL_TRAIN), 854530)
check("evaluation rows", len(EVALUATION_TASKS) == 114869, len(EVALUATION_TASKS), 114869)
check("evaluation query rows", EVALUATION_TASKS["role"].eq("query").sum() == 57569, int(EVALUATION_TASKS["role"].eq("query").sum()), 57569)
check("evaluation query positives", PROTOCOL_SUMMARY["evaluation_query_positives"] == 23501, PROTOCOL_SUMMARY["evaluation_query_positives"], 23501)
check("validation warm blocks", warm_blocks_valid(VALIDATION_TASKS), True, True)
check("evaluation warm blocks", warm_blocks_valid(EVALUATION_TASKS), True, True)
check("validation task partition", task_partition_valid(VALIDATION_TASKS, VALIDATION_ITEMS), True, True)
check("evaluation task partition", task_partition_valid(EVALUATION_TASKS, NEW_ITEMS), True, True)
check("new items absent from final training", not (set(FINAL_TRAIN["item_id"]) & NEW_ITEMS), True, True)
check("validation items absent from tuning", not (set(TUNING_TRAIN["item_id"]) & VALIDATION_ITEMS), True, True)
check("task rows absent from tuning", set(TUNING_TRAIN["source_row"]).isdisjoint(set(VALIDATION_TASKS["source_row"]) | set(EVALUATION_TASKS["source_row"])), True, True)
check("label rule in every split", all(labels_valid(table) for table in (TUNING_TRAIN, FINAL_TRAIN, VALIDATION_TASKS, EVALUATION_TASKS)), True, True)
check("validation query has both labels", VALIDATION_TASKS.loc[VALIDATION_TASKS["role"].eq("query"), "label"].nunique() == 2, 2, 2)
check("evaluation query has both labels", EVALUATION_TASKS.loc[EVALUATION_TASKS["role"].eq("query"), "label"].nunique() == 2, 2, 2)
check("evaluation users visible in tuning", set(EVALUATION_TASKS["user_id"]) <= set(TUNING_TRAIN["user_id"]), True, True)
check("validation users visible in tuning", set(VALIDATION_TASKS["user_id"]) <= set(TUNING_TRAIN["user_id"]), True, True)
check("shared indices bijective and consistent", indices_valid(), True, True)

PROTOCOL_PASS = all(row["status"] == "PASS" for row in PROTOCOL_CHECKS)
display(pd.DataFrame(PROTOCOL_CHECKS))
display(Markdown("### Protocol checks: " + ("PASS" if PROTOCOL_PASS else "FAIL")))

,check,status,observed,expected
0,old item count,PASS,1420,1420
1,new item count,PASS,955,955
2,old validation item count,PASS,284,284
3,old tuning item count,PASS,1136,1136
4,excluded rated item count,PASS,1331,1331
5,unrated item count,PASS,177,177
6,item cohorts disjoint,PASS,True,True
7,tuning rows,PASS,684728,684728
8,final old-item rows,PASS,854530,854530
9,evaluation rows,PASS,114869,114869


### Protocol checks: PASS

## Atomic Protocol Export

Publish an immutable generation and then atomically swap the pointer manifest.
The bundle copies only protocol-participating user/item metadata, so downstream
model notebooks need no separate raw-dataset input.

In [7]:
def json_ready(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item"):
        return value.item()
    return str(value)


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(content, encoding="utf-8")
    temporary.replace(path)


def write_csv(path: Path, table: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    table.to_csv(temporary, index=False, lineterminator="\n")
    temporary.replace(path)


def relative_output(path: Path) -> str:
    return str(path.resolve().relative_to(ARTIFACT_ROOT.resolve()))


def csv_schema(table: pd.DataFrame) -> dict[str, Any]:
    def dtype_name(dtype: Any) -> str:
        name = str(dtype)
        return "string" if name in {"str", "string"} or name.startswith("string") else name

    return {
        "columns": list(table.columns),
        "read_csv_dtypes": {column: dtype_name(dtype) for column, dtype in table.dtypes.items()},
    }


if not PROTOCOL_PASS:
    raise RuntimeError("Protocol checks failed; artifacts were not published")

PROTOCOL_OUTPUT_ROOT.resolve().relative_to(ARTIFACT_ROOT.resolve())
bundle_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:12]
staging_root = PROTOCOL_OUTPUT_ROOT / f".staging-{bundle_id}"
generation_root = PROTOCOL_OUTPUT_ROOT / "generations" / bundle_id
pointer_path = PROTOCOL_OUTPUT_ROOT / "manifest.json"
staging_root.mkdir(parents=True, exist_ok=False)

TABLES = {
    "tuning_train": TUNING_TRAIN,
    "final_train": FINAL_TRAIN,
    "validation_tasks": VALIDATION_TASKS,
    "evaluation_tasks": EVALUATION_TASKS,
    "users": PROTOCOL_USERS,
    "items": PROTOCOL_ITEMS_METADATA,
    "item_cohorts": ITEM_COHORTS,
}
OUTPUT_ARTIFACTS: dict[str, Any] = {}

try:
    for name, table in TABLES.items():
        staging_path = staging_root / f"{name}.csv"
        published_path = generation_root / f"{name}.csv"
        write_csv(staging_path, table)
        OUTPUT_ARTIFACTS[name] = {
            "path": relative_output(published_path),
            "sha256": sha256_file(staging_path),
            "rows": len(table),
        }

    manifest = {
        "protocol_schema_version": CONFIG.schema_version,
        "protocol_status": "PASS",
        "bundle_id": bundle_id,
        "bundle_manifest": relative_output(generation_root / "manifest.json"),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "upstream_audit": {
            "schema_version": AUDIT_MANIFEST["audit_schema_version"],
            "bundle_id": AUDIT_MANIFEST["bundle_id"],
            "pointer_sha256": UPSTREAM_POINTER_SHA256,
        },
        "configuration": CONFIG_PAYLOAD,
        "configuration_sha256": CONFIG_HASH,
        "summary": PROTOCOL_SUMMARY,
        "cohort_summary": COHORT_SUMMARY.to_dict(orient="records"),
        "task_summary": TASK_SUMMARY.to_dict(orient="records"),
        "checks": PROTOCOL_CHECKS,
        "artifacts": OUTPUT_ARTIFACTS,
        "output_schemas": {name: csv_schema(table) for name, table in TABLES.items()},
        "path_contract": {
            "pointer_from_protocol_root": "protocols/ml-1m/coldstart-v1/manifest.json",
            "artifact_paths": "relative to COLDSTART_PROTOCOL_ROOT when consumed downstream",
        },
        "phase_contract": {
            "Cold": {"adaptation_increment": [], "graph_visible_support": []},
            "Warm A": {
                "adaptation_increment": ["warm_a"],
                "graph_visible_support": ["warm_a"],
            },
            "Warm B": {
                "adaptation_increment": ["warm_b"],
                "continue_item_state_from": "Warm A",
                "graph_visible_support": ["warm_a", "warm_b"],
            },
            "Warm C": {
                "adaptation_increment": ["warm_c"],
                "continue_item_state_from": "Warm B",
                "graph_visible_support": ["warm_a", "warm_b", "warm_c"],
            },
            "query": "identical role=query rows in every phase",
        },
        "feature_contract": {
            "allowed_user_features": ["user_id", "gender", "age", "occupation", "zip_code"],
            "allowed_item_features": ["item_id", "title", "genres", "release_year"],
            "administrative_only": [
                "source_row",
                "timestamp",
                "rating",
                "label",
                "item_rank",
                "role",
                "interaction_count",
                "cohort",
                "user_idx",
                "item_idx",
            ],
            "rule": "administrative fields must not be predictive model features",
        },
        "training_contract": {
            "tuning": "tuning_train; validation items excluded",
            "final_refit": "final_train after all choices and thresholds are frozen",
            "candidate_policy": CONFIG.candidate_policy,
            "graph_edge_policy": CONFIG.graph_edge_policy,
            "f1_thresholds": "select per model and phase on validation query rows only",
        },
    }

    manifest_text = json.dumps(json_ready(manifest), indent=2, sort_keys=True) + "\n"
    write_text(staging_root / "manifest.json", manifest_text)
    generation_root.parent.mkdir(parents=True, exist_ok=True)
    staging_root.replace(generation_root)
    write_text(pointer_path, manifest_text)
except Exception:
    if staging_root.exists():
        shutil.rmtree(staging_root, ignore_errors=True)
    raise

HANDOFF_READY = pointer_path.is_file() and pointer_path.read_text() == (
    generation_root / "manifest.json"
).read_text()
show_records(
    [
        {
            "protocol_status": "PASS" if HANDOFF_READY else "FAIL",
            "bundle_id": bundle_id,
            "manifest": str(pointer_path),
            "artifacts": len(OUTPUT_ARTIFACTS),
        }
    ]
)
display(Markdown("### Notebook 02 protocol: " + ("READY" if HANDOFF_READY else "BLOCKED")))

,protocol_status,bundle_id,manifest,artifacts
0,PASS,20260716T201434-04e93786b698,/kaggle/working/artifacts/protocols/ml-1m/cold...,7


### Notebook 02 protocol: READY

## Continue to Model Notebooks

**Notebook 03, LightGCN:** load this protocol manifest, verify hashes, use
`tuning_train` for selection and `final_train` for the frozen final refit. Build
Cold/A/B/C graphs cumulatively from the corresponding task support roles; all
visible interactions are graph edges. Score the same `role=query` rows in every
phase and never drop isolated new-item nodes.

**Notebook 04, EmerG:** use the same base/support/query rows and indexed user/item
metadata. Fit title, genre, ZIP, and categorical vocabularies on tuning-visible
old items only, with PAD/UNK handling. Adapt declared item-specific parameters
sequentially using A, then only the B increment from the retained A state, then
only the C increment from the retained B state. Select per-phase F1 thresholds
on validation query rows before final refit and new-item evaluation.